# Generating new models with GeoSANE

GeoSANE is a geospatial model foundry that learns a shared latent representation from the weights of existing remote sensing foundation models and task-specific models. Instead of training yet another model from geospatial data alone, GeoSANE operates directly in weight space: it learns from the parameters of many pretrained models and uses that shared representation to generate new model weights on demand for a target architecture.

At a high level, generating and evaluating a new model with GeoSANE involves the following steps:
1. Start from a trained GeoSANE autoencoder that has learned a latent representation of a population of geospatial models.
2. Choose a target architecture, such as a timm backbone that you want to instantiate with generated weights.
3. Choose a downstream dataset and task configuration, for example classification, segmentation, or detection.
4. Use GeoSANE to generate candidate model checkpoints in the target model space.
5. Fine-tune the generated checkpoints on the downstream task.

This notebook provides a minimal end-to-end example of that workflow using timm models as prompts. 

Before running the notebook, make sure the repository dependencies are installed and that the checkpoint and dataset paths in the configuration cell below point to valid local files.


## 1. Prepare the environment

This cell locates the local `src` directory, adds it to `sys.path`, and imports the small helper API used throughout the notebook.


In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "shrp").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the repo src directory.")

from shrp.evaluation.experiment_timm_helpers import (
    DEFAULT_CALLBACK_KWARGS,
    TimmExperimentConfig,
    configure_notebook_environment,
    run_experiment_suite,
    validate_experiment_config,
)

REPO_SRC = configure_notebook_environment()
print(f"Repo src: {REPO_SRC}")


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0a0+40ec155e58.nv24.3
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Repo src: /netscratch2/jhanna/wslxrs/geosane-internal


## 2. Create the downstream dataset file

Before running the GeoSANE evaluation, create the downstream dataset `.pt` file referenced by `reference_dataset_path`.

The expected pattern is simple: instantiate the downstream dataset for the train and validation/test splits, store them in a dictionary with keys like `trainset` and `testset`, and save that dictionary with `torch.save`. The main evaluation script will load this file later.

The example below uses the Sen1Floods11 preprocessing pipeline and saves `sen1flood11_preprocessed.pt`, which matches the downstream task configuration used later in this notebook.


In [2]:
import os
from pathlib import Path

import torch

os.environ["RAY_OBJECT_STORE_ALLOW_SLOW_STORAGE"] = "1"

from downstream_datasets.sen1flood11_data import Sen1Floods11HandLabeledDataset

data_path = REPO_SRC.parent / "data"
data_path.mkdir(parents=True, exist_ok=True)

trainset = Sen1Floods11HandLabeledDataset(split="train", resize_to=(224, 224))
testset = Sen1Floods11HandLabeledDataset(split="val", resize_to=(224, 224))

dataset = {
    "trainset": trainset,
    "testset": testset,
}

dataset_pt_path = data_path / "sen1flood11_preprocessed.pt"
torch.save(dataset, dataset_pt_path)

print(len(trainset), len(testset))
print(f"Saved downstream dataset to: {dataset_pt_path}")


341 90
Saved downstream dataset to: /netscratch2/jhanna/wslxrs/data/sen1flood11_preprocessed.pt


## 3. Configure the evaluation

This is the main cell you should edit for your own experiment.

The most important fields are:
- `model_path`: the directory containing a trained GeoSANE checkpoint and its `params.json`
- `reference_dataset_path`: the preprocessed dataset used for downstream evaluation, e.g. eurosat, spacenet etc.
- `model_names`: the model prompt (model name from the timm library)
- `callback_kwargs`: the downstream-task configuration, including dataset name, task type, number of classes, and number of input channels

If you created a new downstream dataset file in the previous step, update `reference_dataset_path` to point to that saved `.pt` file and make sure `callback_kwargs` matches the corresponding task.


In [3]:
EXPERIMENT = TimmExperimentConfig(
    model_path=Path(
        "/ds2/remote_sensing/"
        "AE_trainable_4d8b1_00000_0_sampling_path=ds2_remote_sensing_huggingface_zoos_datasets_2025-08-30_07-24-02"
    ),
    reference_dataset_path=REPO_SRC.parent / "data" / "sen1flood11_preprocessed.pt",
    model_names=[
        "swin_s3_base_224.ms_in1k",
        # "resnet50",
        # "mobilenetv2_100.ra_in1k",
    ],
    checkpoint_rel_path=Path("checkpoint_000300/state.pt"),
    output_dir=None,
    seed=32,
    steps_per_epoch=123,
    device=None,
    callback_kwargs={
        **DEFAULT_CALLBACK_KWARGS,
        "finetuning_epochs": 10,
        "downstream_dataset": "sen1floods11",
        "num_classes": 1,
        "num_channels": 2,
        "task": "segmentation",
    },
)

print(EXPERIMENT)
print(f"Resolved device: {EXPERIMENT.resolved_device()}")
print(f"Resolved output dir: {EXPERIMENT.resolved_output_dir()}")


TimmExperimentConfig(model_path=PosixPath('/netscratch2/jhanna/wslxrs/shrp_finetuned_lr_1e-6/shrp_finetuned_lr_1e-6/AE_trainable_4d8b1_00000_0_sampling_path=ds2_remote_sensing_huggingface_zoos_datasets_2025-08-30_07-24-02'), reference_dataset_path=PosixPath('/netscratch2/jhanna/wslxrs/data/sen1flood11_preprocessed.pt'), model_names=['swin_s3_base_224.ms_in1k'], checkpoint_rel_path=PosixPath('checkpoint_000300/state.pt'), output_dir=None, seed=32, steps_per_epoch=123, device=None, callback_kwargs={'finetuning_epochs': 10, 'repetitions': 1, 'bootstrap_number': 1, 'mode': 'token', 'every_n_epochs': 0, 'eval_iterations': [0], 'batch_size': 16, 'reset_classifier': True, 'halo': True, 'halo_wse': 512, 'halo_hs': 64, 'bn_condition_iters': 50, 'dense': True, 'use_relative_pos': False, 'downstream_dataset': 'sen1floods11', 'num_classes': 1, 'num_channels': 2, 'task': 'segmentation', 'linear_probing': False})
Resolved device: cuda
Resolved output dir: /netscratch2/jhanna/wslxrs/shrp_finetuned_lr

In [4]:
validate_experiment_config(EXPERIMENT)
print("Experiment config looks valid.")


Experiment config looks valid.


## 4. Run the evaluation pipeline

This cell loads the trained GeoSANE autoencoder, constructs the timm evaluation callback for each requested model, and runs the downstream evaluation. One JSON results file is written per evaluated timm backbone.

In [ ]:
all_results = run_experiment_suite(EXPERIMENT, iteration=0)
print(f"Finished {len(all_results)} model evaluations.")


INFO:Initialize Model
Seed set to 32
INFO:device: cuda


Using functional sinusoidal position embeddings


INFO:compiling the model... (takes a ~minute)
INFO:compiled successfully


model: use simclr NT_Xent loss
Running single-gpu. send model to device: cuda
num decayed parameter tensors: 134, with 909,036,420 parameters
num non-decayed parameter tensors: 72, with 101,854 parameters
using fused AdamW: False
++++++ USE AUTOMATIC MIXED PRECISION +++++++


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
INFO:Evaluating swin_s3_base_224.ms_in1k
INFO:sampling:: create reference checkoint
INFO:Loading pretrained weights from Hugging Face hub (timm/swin_s3_base_224.ms_in1k)
INFO:HTTP Request: HEAD https://huggingface.co/timm/swin_s3_base_224.ms_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO:[timm/swin_s3_base_224.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:sampling:: get first anchor embeddings from anchor_tokenized/swin_s3_base_224.ms_in1k_dataset_tk_230.pt
INFO:get anchor embeddings from anchor_tokenized/swin_s3_base_224.ms_in1k_dataset_tk_230.pt
INFO:haloify embeddings


get anchor embeddings with halo: True window size: 512 halo size: 64


INFO:unhaloify embeddings
INFO:sampling:: sample models
INFO:sampling models using anchor_z: torch.Size([1, 308184, 128]) (torch.float16) - anchor_pos: torch.Size([308184, 3]) (torch.int32) and anchor_w_shape: torch.Size([1, 308184, 230]), anchor_types: None


unstack haloed batches


## 5. Inspect the outputs

After the evaluation finishes, this cell lists the generated results files and previews the first saved JSON file. This is a quick sanity check that the run completed and that the metrics were written correctly.


## Additional notes

- This notebook is an example workflow, not the only supported way to run SHRP evaluations.
- To compare multiple timm backbones, update only `model_names`.
- To switch to a different downstream dataset or task setup, update only `callback_kwargs`.
- To evaluate a different trained SHRP model, update `model_path` and, if needed, `checkpoint_rel_path`.
- The reusable logic lives in `shrp.evaluation.experiment_timm_helpers`, which keeps this notebook short, readable, and suitable for inclusion in the repository.
